# Certificate Name Autofiller

This notebook automates the process of adding names to a certificate template, uploading them to Google Drive, and retrieving shareable links.

### Features:
- Bulk processing from an Excel file (`.xlsx`).
- Automatic text placement and styling (calibrated for OPH Template).
- Google Drive integration for storage and link generation.
- Updates the original dataset with matching certificate links.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from google.colab import files
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration
Set the paths for your template, dataset, and the output folder on Google Drive.

In [ ]:
# Paths (Update these based on your Google Drive structure)
FOLDER_NAME = 'your folder name'
BASE_PATH = f'/content/drive/MyDrive/{FOLDER_NAME}'
TEMPLATE_PATH = os.path.join(BASE_PATH, 'your template.png')
DATASET_PATH = os.path.join(BASE_PATH, 'your dataset.xlsx')
# OUTPUT_FOLDER_NAME = 'Generated_Certificates'
NAME = 'name' #columns name title


# Font Configuration (Calibrated for the provided template)
FONT_PATH = os.path.join(BASE_PATH, 'your font.ttf') # Available on Colab
FONT_SIZE = 110
TEXT_COLOR = (35, 35, 35) # Dark gray/black
TEXT_Y_POSITION = 611 # Vertical center
IMAGE_WIDTH = 2000 # Template width

## Utility Functions

In [ ]:
def create_certificate(name, template_path, output_path, font_path, font_size, color, y_pos, img_width):
    """Draws the name on the certificate template."""
    img = Image.open(template_path).convert('RGB')
    draw = ImageDraw.Draw(img)

    try:
        font = ImageFont.truetype(font_path, font_size)
    except:
        print(f"Font {font_path} not found, using default.")
        font = ImageFont.load_default()

    # Calculate text width to center it
    bbox = draw.textbbox((0, 0), name, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    x_pos = (img_width - text_width) / 2
    # Adjust y_pos to be the center of the text
    adjusted_y = y_pos - (text_height / 2)

    draw.text((x_pos, adjusted_y), name, fill=color, font=font)
    img.save(output_path)
    return output_path

def get_drive_service():
    """Authenticate and return Google Drive service."""
    from google.colab import auth
    auth.authenticate_user()
    return build('drive', 'v3')

def create_drive_folder(service, folder_name):
    """Creates a folder on Google Drive and returns its ID."""
    file_metadata = {
        'name': folder_name,
        'mimeType': 'application/vnd.google-apps.folder'
    }
    file = service.files().create(body=file_metadata, fields='id').execute()
    return file.get('id')

def upload_and_get_link(service, file_path, folder_id):
    """Uploads a file to Drive and returns a shareable view link."""
    file_metadata = {
        'name': os.path.basename(file_path),
        'parents': [folder_id]
    }
    media = MediaFileUpload(file_path, mimetype='image/png')
    file = service.files().create(body=file_metadata, media_body=media, fields='id').execute()

    file_id = file.get('id')

    # Make file public (view access for anyone with link)
    service.permissions().create(
        fileId=file_id,
        body={'type': 'anyone', 'role': 'reader'}
    ).execute()

    # Retrieve the link
    res = service.files().get(fileId=file_id, fields='webViewLink').execute()
    return res.get('webViewLink')

# Helper to find folder ID on Google Drive by its path
def find_folder_id_by_path(service, drive_path, max_retries=5, retry_delay_seconds=5):
    """Finds the Google Drive folder ID for a given absolute path starting from MyDrive, with retries."""

    # Split the path to get components relative to MyDrive
    parts = drive_path.split('/MyDrive/')
    if len(parts) < 2:
        print("Invalid Drive path format. Expected '/content/drive/MyDrive/...' ")
        return None

    relative_path = parts[1]
    path_components = relative_path.split('/')

    current_parent_id = 'root' # 'root' refers to 'My Drive'

    for component_index, component in enumerate(path_components):
        if not component: # Skip empty components if any
            continue

        found_component = False
        for attempt in range(max_retries):
            query = f"name = '{component}' and '{current_parent_id}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
            try:
                response = service.files().list(q=query, spaces='drive', fields='files(id, name)').execute()
                folders = response.get('files', [])
                if folders:
                    current_parent_id = folders[0]['id']
                    found_component = True
                    break # Found, move to next component
            except Exception as e:
                print(f"Error during Drive API call for folder '{component}' (attempt {attempt + 1}): {e}")

            if not found_component and attempt < max_retries - 1:
                print(f"Folder '{component}' not found. Retrying in {retry_delay_seconds} seconds...")
                time.sleep(retry_delay_seconds)

        if not found_component:
            print(f"Folder '{component}' not found in path {drive_path} after multiple retries. Current parent ID: {current_parent_id}")
            return None
    return current_parent_id

# Helper to get link for an existing file and ensure it's shareable
def get_shareable_link_for_existing_file(service, file_name, parent_folder_id):
    """Gets the shareable webViewLink for an existing file in a specific Google Drive folder and makes it public."""
    query = f"name = '{file_name}' and '{parent_folder_id}' in parents and trashed = false"
    response = service.files().list(q=query, spaces='drive', fields='files(id, webViewLink)').execute()

    files = response.get('files', [])
    if not files:
        return None

    file_id = files[0]['id']
    web_view_link = files[0].get('webViewLink')

    # Ensure the file is shareable (publicly readable)
    try:
        permissions_response = service.permissions().list(fileId=file_id, fields='permissions(type,role)').execute()
        is_public = False
        for p in permissions_response.get('permissions', []):
            if p['type'] == 'anyone' and p['role'] == 'reader':
                is_public = True
                break

        if not is_public:
            # Make file public if it's not already
            service.permissions().create(
                fileId=file_id,
                body={'type': 'anyone', 'role': 'reader'},
                fields='id'
            ).execute()
            # Re-fetch link, as sometimes it might change after permission update
            file_info = service.files().get(fileId=file_id, fields='webViewLink').execute()
            web_view_link = file_info.get('webViewLink')
    except Exception as e:
        print(f"Error checking/setting permissions for {file_name} ({file_id}): {e}")
        return None # Return None if permission setting fails

    return web_view_link

## Bulk Processing
This step will read the Excel file, generate certificates, and update the dataset.

In [ ]:
from tqdm.notebook import tqdm

# 1. Initialize Service
service = get_drive_service()

# # 2. Create Output Folder on Drive
# # Note: This will create a new folder each time if a folder with the same name already exists.
# folder_id = create_drive_folder(service, OUTPUT_FOLDER_NAME)
# print(f"Created Folder ID: {folder_id}")

# 3. Read Dataset
df = pd.read_excel(DATASET_PATH)
if 'Certificate_Link' not in df.columns:
    df['Certificate_Link'] = ""

temp_dir = 'temp_certs'
if not os.path.exists(temp_dir):
    os.makedirs(temp_dir)

# List to store local paths and corresponding DataFrame indices
certs_to_upload = []


In [ ]:
df[f'{NAME}'] = df[f'{NAME}'].fillna('ไม่กรอกชื่อ-สกุล')
assert df[f'{NAME}'].isna().sum() == 0
df

In [ ]:
# #rollback
# !rm -rf temp_certs/*

In [ ]:

# 4. First Pass: Generate all certificates locally
print("\nGenerating certificates locally...")
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Generating local certificates"):
  name = str(row[f'{NAME}'])
    # If name is empty, 'nan' string (from empty Excel cells), or actual NaN, replace with 'undefined'
  if not name or name == 'nan':
      name = 'undefined'

  # a. Create Certificate Locally
  clean_name = "".join([c for c in name if c.isalnum() or c == ' ']).rstrip()
  # Add order numbering to the filename
  file_name = f"{index + 1:04d}_Certificate_{clean_name.replace(' ', '_')}.png"
  local_path = os.path.join(temp_dir, file_name)

  try:
      create_certificate(name, TEMPLATE_PATH, local_path, FONT_PATH, FONT_SIZE, TEXT_COLOR, TEXT_Y_POSITION, IMAGE_WIDTH)
      certs_to_upload.append({'index': index, 'name': name, 'local_path': local_path})
  except Exception as e:
      print(f"Error generating local certificate for {name}: {e}")

# # 5. Second Pass: Upload certificates to Google Drive and get links
# print("\nUploading certificates to Google Drive and getting links...")
# for item in tqdm(certs_to_upload, desc="Uploading to Drive"):
#     index = item['index']
#     name = item['name']
#     local_path = item['local_path']

#     # b. Upload and Get Link only if not already present
#     # This allows for resuming if the process was interrupted and some links were already generated
#     if pd.isna(df.at[index, 'Certificate_Link']) or df.at[index, 'Certificate_Link'] == "":
#         try:
#             link = upload_and_get_link(service, local_path, folder_id)
#             df.at[index, 'Certificate_Link'] = link
#         except Exception as e:
#             print(f"Error uploading {name} from {local_path}: {e}")
#     # else:
#     #     print(f"Skipping {name}: Link already exists.") # Can uncomment for more verbose logging

#     # Clean up local file after upload attempt
#     if os.path.exists(local_path):
#         os.remove(local_path)

# print("\nLocal temporary certificates removed.")

# # 6. Save Updated Dataset
# updated_dataset_path = DATASET_PATH.replace('.xlsx', '_updated.xlsx')
# df.to_excel(updated_dataset_path, index=False)
# print(f"\nAll done! Updated dataset saved to: {updated_dataset_path}")

In [ ]:
#value passing checker
!echo $FOLDER_NAME
assert FOLDER_NAME != None

In [ ]:
!zip -r temp_certs.zip -r temp_certs

In [ ]:
!ls drive/MyDrive/$FOLDER_NAME

In [ ]:
!cp temp_certs.zip drive/MyDrive/$FOLDER_NAME

In [ ]:
!unzip /content/drive/MyDrive/$FOLDER_NAME/temp_certs.zip -d /content/drive/MyDrive/$FOLDER_NAME

In [ ]:
!ls /content/drive/MyDrive/$FOLDER_NAME/temp_certs | wc -l

In [ ]:
import os
from googleapiclient.discovery import build
from google.colab import auth
import pandas as pd

print("Retrieving shared links and updating DataFrame...")

# Ensure Google Drive service is authenticated
try:
    service = get_drive_service()
except NameError:
    print("Error: 'get_drive_service' function not found. Please run the 'Utility Functions' section first.")
    auth.authenticate_user()
    service = build('drive', 'v3')

# Define the target folder path on Google Drive
target_drive_folder_path = f'/content/drive/MyDrive/{FOLDER_NAME}/temp_certs'

# Find the Google Drive folder ID using the helper function
try:
    certs_drive_folder_id = find_folder_id_by_path(service, target_drive_folder_path)
except NameError:
    print("Error: 'find_folder_id_by_path' function not found. Please run the 'Utility Functions' section first.")
    certs_drive_folder_id = None

if not certs_drive_folder_id:
    print(f"Could not find the Google Drive folder: {target_drive_folder_path}. Please ensure the path is correct and the folder exists.")
else:
    print(f"Found certificates Drive folder ID: {certs_drive_folder_id}")

    # List all files in the Drive folder
    query = f"'{certs_drive_folder_id}' in parents and trashed = false and mimeType = 'image/png'"
    all_drive_files = []
    page_token = None
    while True:
        response = service.files().list(q=query, spaces='drive', fields='nextPageToken, files(id, name, webViewLink)', pageSize=1000, pageToken=page_token).execute()
        all_drive_files.extend(response.get('files', []))
        page_token = response.get('nextPageToken')
        if not page_token:
            break

    print(f"Found {len(all_drive_files)} PNG files in the Google Drive folder.")

    links_from_drive = {}
    present_indices_from_drive = set()

    for drive_file in all_drive_files:
        file_name = drive_file['name']
        web_view_link = drive_file.get('webViewLink') # Get the link directly if available

        parts = file_name.split('_Certificate_', 1)
        if len(parts) > 1:
            try:
                extracted_index_plus_one = int(parts[0])
                original_df_index = extracted_index_plus_one - 1

                # If webViewLink is not directly available or to ensure it's shareable
                if not web_view_link:
                    # Use get_shareable_link_for_existing_file to get/ensure shareability
                    web_view_link = get_shareable_link_for_existing_file(service, file_name, certs_drive_folder_id)

                if web_view_link:
                    links_from_drive[original_df_index] = web_view_link
                    present_indices_from_drive.add(original_df_index)
                else:
                    print(f"Warning: Could not get shareable link for {file_name}")

            except ValueError:
                print(f"Warning: Could not parse index from filename: {file_name}")

    # Update the DataFrame with retrieved links
    for idx, link in links_from_drive.items():
        if idx < len(df): # Ensure the index is within DataFrame bounds
            df.at[idx, 'Certificate_Link'] = link

    print(f"Updated DataFrame with {len(links_from_drive)} certificate links from Google Drive.")

    # Get all expected indices from the DataFrame
    all_expected_indices = set(df.index)

    # Identify missing indices (those in df but not in Drive files processed)
    missing_indices_by_filename_check = sorted(list(all_expected_indices - present_indices_from_drive))

    print(f"\nTotal expected certificates (from df): {len(all_expected_indices)}")
    print(f"Certificates found in Drive and processed: {len(present_indices_from_drive)}")
    print(f"Missing certificates (indices in df but no corresponding file found/processed in Drive): {len(missing_indices_by_filename_check)}")

    if missing_indices_by_filename_check:
        print("\nIndices of missing certificates based on filename check:")
        print(missing_indices_by_filename_check)
        print("\nDetails for these missing certificates:")
        try:
             display(df.loc[missing_indices_by_filename_check, [f'{NAME}', 'Certificate_Link']])
        except NameError:
             print("Error: 'NAME' variable not defined. Please run the configuration cell (IzOPp6eOvAXR) first.")
             # Fallback to a common column name if NAME is not defined for display purposes
             display(df.loc[missing_indices_by_filename_check, [f'{NAME}', 'Certificate_Link']])

        # You can now assign this list to manual_missing_indices for the next step
        globals()['manual_missing_indices'] = missing_indices_by_filename_check
        print("\n'manual_missing_indices' variable updated with the list of missing indices.")
    else:
        print("\nAll expected certificates are present and links updated in the DataFrame from the Google Drive folder according to filename parsing!")

In [ ]:
assert df['Certificate_Link'].isna().sum() == 0

In [ ]:
processed_df = df
processed_df

In [ ]:
batch_name = 'ADDITIONAL_LIST'

In [ ]:
processed_df.to_excel(f'processed_data_{batch_name}.xlsx')